# Phase 1 — Data Loading & Preparation
**Goal:** Load all AWV fietstellingen data, merge the three datasets, filter for cyclists, engineer temporal features, and save a clean processed file.

> Run `download_data.py` first to populate `data/raw/` before executing this notebook.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
import holidays
import warnings
warnings.filterwarnings('ignore')

RAW_DIR     = Path('../data/raw')
COUNTS_DIR  = RAW_DIR / 'counts'
PROC_DIR    = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

print('Raw data folder exists:', RAW_DIR.exists())
print('Number of monthly count files found:',
      len(list(COUNTS_DIR.glob('data-*.csv'))))

## 1. Load Metadata (sites + directions)

In [ ]:
sites = pd.read_csv(RAW_DIR / 'sites.csv', sep=';')
sites.columns = sites.columns.str.strip().str.lower().str.replace(' ', '_')
print('sites.csv shape:', sites.shape)
sites.head()

In [ ]:
richtingen = pd.read_csv(RAW_DIR / 'richtingen.csv', sep=';')
richtingen.columns = richtingen.columns.str.strip().str.lower().str.replace(' ', '_')
print('richtingen.csv shape:', richtingen.shape)
richtingen.head()

## 2. Load & Concatenate Monthly Count Files (2020–2025)

In [ ]:
count_files = sorted(COUNTS_DIR.glob('data-*.csv'))
print(f'Loading {len(count_files)} monthly files …')

chunks = []
for fp in tqdm(count_files, desc='Reading CSVs'):
    try:
        df = pd.read_csv(fp, sep=';', low_memory=False)
        df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
        chunks.append(df)
    except Exception as e:
        print(f'  Warning: could not read {fp.name}: {e}')

counts_raw = pd.concat(chunks, ignore_index=True)
print(f'Total rows loaded: {len(counts_raw):,}')
counts_raw.head()

## 3. Filter for Cyclists (`type = FIETSERS`)

In [ ]:
print('Unique type values:')
print(counts_raw['type'].value_counts())

counts = counts_raw[counts_raw['type'].str.upper() == 'FIETSERS'].copy()
print(f'Rows after filtering for FIETSERS: {len(counts):,}')
print(f'Rows dropped: {len(counts_raw) - len(counts):,}')

## 4. Parse Timestamps & Remove Duplicates

In [ ]:
counts['van'] = pd.to_datetime(counts['van'], errors='coerce')
counts['tot'] = pd.to_datetime(counts['tot'], errors='coerce')

n_before = len(counts)
counts.dropna(subset=['van', 'tot', 'aantal'], inplace=True)
counts['aantal'] = pd.to_numeric(counts['aantal'], errors='coerce')
counts.dropna(subset=['aantal'], inplace=True)
counts = counts[counts['aantal'] >= 0]
counts.drop_duplicates(subset=['site_id', 'richting', 'van'], inplace=True)

print(f'Rows before cleaning : {n_before:,}')
print(f'Rows after  cleaning : {len(counts):,}')
print(f'Date range: {counts["van"].min()} → {counts["van"].max()}')

## 5. Merge with Sites & Directions Metadata

In [ ]:
for df in [sites, richtingen]:
    for col in df.columns:
        if 'site' in col and 'id' in col:
            df.rename(columns={col: 'site_id'}, inplace=True)
            break

df = counts.merge(sites, on='site_id', how='left', suffixes=('', '_site'))
df = df.merge(richtingen, on=['site_id', 'richting'], how='left', suffixes=('', '_dir'))
print('Merged dataframe shape:', df.shape)

## 6. Temporal Feature Engineering

In [ ]:
be_holidays = holidays.Belgium(years=range(2020, 2026))

df['hour']       = df['van'].dt.hour
df['day_of_week']= df['van'].dt.dayofweek
df['day_name']   = df['van'].dt.day_name()
df['month']      = df['van'].dt.month
df['year']       = df['van'].dt.year
df['date']       = df['van'].dt.date
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_holiday'] = df['date'].astype(str).map(lambda d: 1 if d in be_holidays else 0)

def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                     return 'Autumn'

df['season'] = df['month'].map(get_season)
print(df[['van','hour','day_of_week','month','year','is_weekend','is_holiday','season']].head(10))

In [ ]:
df.sort_values(['site_id', 'van'], inplace=True)
df['rolling_7d_avg'] = (
    df.groupby('site_id')['aantal']
    .transform(lambda x: x.rolling(window=7*24, min_periods=1).mean())
)
print('Rolling average computed.')

## 7. Data Quality Assessment

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0])

site_counts = df.groupby('site_id')['aantal'].count()
expected = 6 * 365 * 24
completeness = (site_counts / expected).clip(upper=1.0)
good_sites = completeness[completeness >= 0.80].index
print(f'Sites with >= 80% completeness: {len(good_sites)} / {len(site_counts)}')

df_clean = df[df['site_id'].isin(good_sites)].copy()
print(f'Rows after quality filter: {len(df_clean):,}')

## 8. Save Processed Data

In [ ]:
out_path = PROC_DIR / 'cyclists_clean.parquet'
df_clean.to_parquet(out_path, index=False)
print(f'Saved to: {out_path}')
print(f'Final shape: {df_clean.shape}')

## ✅ Phase 1 Complete
The processed file `data/processed/cyclists_clean.parquet` is ready for:
- `02_eda.ipynb` — Exploratory visualisations
- `03_clustering.ipynb` — Site classification
- `04_weather_modeling.ipynb` — Weather sensitivity modelling